In [1]:
import os
import sys

# Change directory to use_case (parent directory)
os.chdir('C:/Users/as_cu/Desktop/use_case')


In [2]:
from scripts.data_loader import DataLoader
from scripts.data_cleaner import DataCleaner
from scripts.preprocessor import PreProcessor
from scripts.visualizer import Visualizer
from scripts.formatter import Formatter
from scripts.feature_importance import FeatureImportance
import pandas as pd

In [3]:
# Load all data
data_loader = DataLoader()
onboarding_data = data_loader.load("d0_behaviour_br.csv")
retention_data = data_loader.load("retention_br.csv")

Data loaded successfully. Shape: (112770, 10)
Data loaded successfully. Shape: (104162, 6)


In [4]:
# Clean both datasets
retention_columns = ['d0', 'd3', 'd7', 'd14', 'd30']
data_cleaner = DataCleaner(onboarding_dataset=onboarding_data, retention_dataset=retention_data)
onboarding_data_cleaned, retention_data_cleaned = data_cleaner.retrieve_datasets_cleanse(retention_columns)





-- Managing negative outliers for column: session_length
Dropped 1 rows

-- Managing nulls for quantitative column: age
Number of null values: 30824
Filled with average value: 17

-- Managing nulls for quantitative column: session_length
Number of null values: 10776
Filled with average value: 2134

-- Managing nulls for qualitative column: country
Number of null values: 2
Filled nulls with most frequent value: US

-- Managing nulls for qualitative column: locale
Number of null values: 13
Filled nulls with most frequent value: en-US

-- Managing nulls for qualitative column: device_model
Number of null values: 1
Filled nulls with most frequent value: iPhone12,1

-- Managing nulls for OS column using device_model
Number of null values: 6
Replaced 6 null(s) with iOS (found iPhone or iPad in device_model)
Replaced 0 null(s) with Android (the rest)

-- Managing nulls for binary column: had_meaningful
Number of null values: 40446
Filled nulls with randomly generated binary values with a pro

In [5]:
# Preprocess: merge datasets and add segmented columns
preprocessor = PreProcessor()
merged_dataset = preprocessor.merge(onboarding_data_cleaned, retention_data_cleaned)
merged_dataset[retention_columns] = merged_dataset[retention_columns].fillna(0)
merged_dataset = preprocessor.age_segmentation(merged_dataset)
merged_dataset = preprocessor.session_length_segmentation(merged_dataset)




In [6]:
# Visualizations
visualizer = Visualizer(merged_dataset)
retention_fig = visualizer.retention_comparison()

# Define target variable (can be changed to 'd7', 'd3', etc.)
target_variable = 'd30'
qual_figs = visualizer.qualitative_variables_impacts_on_retention(['country', 'os', 'locale', 'device_model','had_meaningful'], target_variable)

In [7]:
quant_figs = visualizer.quantitative_variable_impact_on_retention(target_var=target_variable)

In [8]:
# Chi-square tests table
formatter = Formatter(merged_dataset)
chi2_results = formatter.chi2_tests_table(['age_segmented', 'session_length_segmented', 'had_meaningful', 'os', 'country'], target_variable)
chi2_results

,Variable,Chi2 P-Value,Is Significant
0,age_segmented,8.411537e-286,Yes
1,session_length_segmented,0.000000e+00,Yes
2,had_meaningful,3.830535e-153,Yes
3,os,8.552338e-199,Yes
4,country,2.485734e-45,Yes


In [9]:
# Correlation study: heatmaps
correlation_figs = visualizer.correlation_study()


In [10]:
# Feature importance using multiple models
feature_importance = FeatureImportance(merged_dataset)
variables_to_test = ['country', 'os', 'age', 'session_length','had_meaningful']
importance_results, roc_fig = feature_importance.compute_feature_importance(variables_to_test, target_var=target_variable)
importance_results



Best Model: LightGBM
ROC AUC Score: 0.6989


,Feature,Importance
0,session_length,1361
1,country,686
2,age,642
3,had_meaningful,163
4,os,148


In [12]:
# Export all graphs to PDF (including retention comparison, heatmaps, and ROC curve)
all_figs = [retention_fig] + qual_figs + quant_figs + correlation_figs + [roc_fig]
formatter.export_to_pdf(all_figs)


Image contains an alpha channel. Computing a separate soft mask (/SMask) image to store transparency in PDF.
Image contains an alpha channel. Computing a separate soft mask (/SMask) image to store transparency in PDF.
Image contains an alpha channel. Computing a separate soft mask (/SMask) image to store transparency in PDF.
Image contains an alpha channel. Computing a separate soft mask (/SMask) image to store transparency in PDF.
Image contains an alpha channel. Computing a separate soft mask (/SMask) image to store transparency in PDF.
Image contains an alpha channel. Computing a separate soft mask (/SMask) image to store transparency in PDF.
Image contains an alpha channel. Computing a separate soft mask (/SMask) image to store transparency in PDF.
Image contains an alpha channel. Computing a separate soft mask (/SMask) image to store transparency in PDF.
Image contains an alpha channel. Computing a separate soft mask (/SMask) image to store transparency in PDF.
Image contains an a

PDF exported to: C:/Users/as_cu/Desktop/use_case/output\churn_risk_analysis.pdf
